# Experiment: Embedding and Cover Artifact Generation

Objective:
- Ensure real + ML covers are available and merged.
- Build payload and stego manifests for all conditions.
- Run embedding stage in dry-run or execute mode.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


def read_rows_csv(path: Path) -> list[dict[str, str]]:
    with path.open('r', newline='', encoding='utf-8') as f:
        return [dict(r) for r in csv.DictReader(f)]


def count_rows(path: Path) -> int:
    return len(read_rows_csv(path))


## Parameters

In [ ]:
from src.data.download_real_covers import download_real_covers
from src.data.generate_ml_covers import generate_ml_covers_from_prompts
from src.data.merge_covers_master import merge_covers_master
from src.pipeline.config import PipelineConfig
from src.pipeline.runner import PipelineRunner

RUN_REAL_DOWNLOAD = False
RUN_ML_GENERATION = False
ML_ENGINE = 'stub'  # switch to 'diffusers' for real generation
EXECUTE_EMBEDDING = False

EXPECTED_GROUPS = 500
PROMPTS_CSV = PROJECT_ROOT / 'data/manifests/generation_prompts.csv'
REAL_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_real.csv'
ML_A_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_ml_a.csv'
ML_B_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_ml_b.csv'
COVERS_MASTER = PROJECT_ROOT / 'data/manifests/covers_master.csv'


## Stage A/B/C: Real Covers, ML Covers, and Merge

In [ ]:
if RUN_REAL_DOWNLOAD:
    outputs_real = download_real_covers(project_root=PROJECT_ROOT)
    print('Real download outputs:')
    for k, v in outputs_real.items():
        print(f'  {k}: {v}')
else:
    print('RUN_REAL_DOWNLOAD=False. Using existing real manifests if present.')

if RUN_ML_GENERATION:
    outputs_ml = generate_ml_covers_from_prompts(
        project_root=PROJECT_ROOT,
        prompts_csv=PROMPTS_CSV,
        engine=ML_ENGINE,
        max_groups=None,
    )
    print('ML generation outputs:')
    for k, v in outputs_ml.items():
        print(f'  {k}: {v}')
else:
    print('RUN_ML_GENERATION=False. Using existing ML manifests if present.')

if REAL_MANIFEST.exists() and ML_A_MANIFEST.exists() and ML_B_MANIFEST.exists():
    merged_path = merge_covers_master(
        project_root=PROJECT_ROOT,
        real_manifest=REAL_MANIFEST,
        ml_a_manifest=ML_A_MANIFEST,
        ml_b_manifest=ML_B_MANIFEST,
        output_manifest=COVERS_MASTER,
        expected_groups=EXPECTED_GROUPS,
    )
    print(f'Merged covers manifest: {merged_path}')
else:
    print('Cannot merge yet; one or more source manifests are missing.')


## Stage D/E: Payload and Stego Manifest + Embedding

In [ ]:
cfg = PipelineConfig.from_project_root(PROJECT_ROOT)
runner = PipelineRunner(cfg)
runner.init_layout()

if not COVERS_MASTER.exists():
    raise FileNotFoundError(f'Missing covers master manifest: {COVERS_MASTER}')

payload_manifest = runner.build_payload_manifest(
    covers_manifest_path=COVERS_MASTER,
    write_payload_files=EXECUTE_EMBEDDING,
)
stego_manifest = runner.build_stego_manifest(
    covers_manifest_path=COVERS_MASTER,
    payload_manifest_path=payload_manifest,
)
embedding_rows = runner.run_embedding_stage(
    stego_manifest_path=stego_manifest,
    execute=EXECUTE_EMBEDDING,
)

print(f'Payload manifest: {payload_manifest}')
print(f'Stego manifest: {stego_manifest}')
print(f'Embedding rows processed: {embedding_rows}')


In [ ]:
print('Cardinality checks:')
print(f'  covers_master rows: {count_rows(COVERS_MASTER)}')
print(f'  payload_manifest rows: {count_rows(PROJECT_ROOT / "data/manifests/payload_manifest.csv")}')
print(f'  stego_manifest rows: {count_rows(PROJECT_ROOT / "data/manifests/stego_manifest.csv")}')

stego_rows = read_rows_csv(PROJECT_ROOT / 'data/manifests/stego_manifest.csv')
print('Sample stego rows:')
for row in stego_rows[:3]:
    print(row['stego_path'])
